🗄️ PHASE 2: SQL Analysis

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator

# Load all datasets 
DATA_PATH = '/Users/tizianogallo/Desktop/Github/f1-analysis/data/'

races        = pd.read_csv(DATA_PATH + 'races.csv')
results      = pd.read_csv(DATA_PATH + 'results.csv')
drivers      = pd.read_csv(DATA_PATH + 'drivers.csv')
constructors = pd.read_csv(DATA_PATH + 'constructors.csv')
lap_times    = pd.read_csv(DATA_PATH + 'lap_times.csv')
pit_stops    = pd.read_csv(DATA_PATH + 'pit_stops.csv')
qualifying   = pd.read_csv(DATA_PATH + 'qualifying.csv')
standings    = pd.read_csv(DATA_PATH + 'driver_standings.csv')
circuits     = pd.read_csv(DATA_PATH + 'circuits.csv')

# Quick sanity check
for name, df in zip(['races','results','drivers','constructors'], 
                    [races, results, drivers, constructors]):
    print(f"{name}: {df.shape} | NaN%: {df.isnull().mean().mean():.2%}")

races: (1125, 18) | NaN%: 0.00%
results: (26759, 18) | NaN%: 0.00%
drivers: (861, 9) | NaN%: 0.00%
constructors: (212, 5) | NaN%: 0.00%


In [4]:
# Replace '\\N' (Ergast null marker) with NaN
for df in [results, qualifying, lap_times, pit_stops]:
    df.replace('\\N', np.nan, inplace=True)

# Parse numeric columns
results['position']      = pd.to_numeric(results['positionOrder'], errors='coerce')
results['grid']          = pd.to_numeric(results['grid'], errors='coerce')
results['points']        = pd.to_numeric(results['points'], errors='coerce')
results['milliseconds']  = pd.to_numeric(results['milliseconds'], errors='coerce')
lap_times['milliseconds']= pd.to_numeric(lap_times['milliseconds'], errors='coerce')

# Parse qualifying times to milliseconds
def time_to_ms(t):
    try:
        m, s = str(t).split(':')
        return int(m) * 60000 + float(s) * 1000
    except:
        return np.nan

for col in ['q1', 'q2', 'q3']:
    qualifying[f'{col}_ms'] = qualifying[col].apply(time_to_ms)

# Add driver full name
drivers['full_name'] = drivers['forename'] + ' ' + drivers['surname']

# Merge year into results
results = results.merge(races[['raceId', 'year', 'circuitId']], on='raceId')

2.1 Set Up SQLite

In [5]:
import sqlite3

conn = sqlite3.connect('/Users/tizianogallo/Desktop/Github/f1-analysis/f1_analysis.db')

# Load all CSVs into SQLite
tables = {
    'races': races, 'results': results, 'drivers': drivers,
    'constructors': constructors, 'lap_times': lap_times,
    'pit_stops': pit_stops, 'qualifying': qualifying,
    'driver_standings': standings, 'circuits': circuits
}
for table_name, df in tables.items():
    df.to_sql(table_name, conn, if_exists='replace', index=False)

print("Database loaded successfully.")

Database loaded successfully.


2.2 Core SQL Queries

In [11]:
def run_query(sql, conn=conn):
    return pd.read_sql_query(sql, conn)

In [18]:
# 1. All-time driver wins leaderboard
df_wins = run_query("""
    SELECT 
        d.forename || ' ' || d.surname AS driver,
        d.nationality,
        COUNT(*) AS wins
    FROM results r
        JOIN drivers d ON r.driverId = d.driverId
    WHERE r.positionOrder = 1
    GROUP BY d.driverId
    ORDER BY wins DESC
    LIMIT 15;
""")
df_wins

,driver,nationality,wins
0,Lewis Hamilton,British,105
1,Michael Schumacher,German,91
2,Max Verstappen,Dutch,63
3,Sebastian Vettel,German,53
4,Alain Prost,French,51
5,Ayrton Senna,Brazilian,41
6,Fernando Alonso,Spanish,32
7,Nigel Mansell,British,31
8,Jackie Stewart,British,27
9,Jim Clark,British,25


In [19]:
# 2. Average pit stop duration by team (modern era 2011+)

df_wins = run_query("""
    SELECT 
        c.name AS constructor,
        ROUND(AVG(p.milliseconds) / 1000.0, 2) AS avg_pit_seconds,
        COUNT(*) AS total_stops
    FROM pit_stops p
        JOIN results r ON p.raceId = r.raceId AND p.driverId = r.driverId
        JOIN constructors c ON r.constructorId = c.constructorId
        JOIN races ra ON p.raceId = ra.raceId
    WHERE ra.year >= 2011 AND p.milliseconds IS NOT NULL
    GROUP BY c.constructorId
    HAVING total_stops > 100
    ORDER BY avg_pit_seconds ASC;
""")
df_wins

,constructor,avg_pit_seconds,total_stops
0,Lotus F1,32.46,285
1,HRT,32.68,150
2,Caterham,33.92,241
3,Marussia,34.40,233
4,Toro Rosso,44.49,683
5,Force India,50.85,609
6,Sauber,57.25,708
7,Renault,63.26,403
8,Williams,76.77,1110
9,Manor Marussia,84.03,156


In [20]:
# 3. Grid-to-win conversion rate by driver (min 50 pole positions)
df_wins = run_query("""
    SELECT
        d.forename || ' ' || d.surname AS driver,
        SUM(CASE WHEN r.grid = 1 THEN 1 ELSE 0 END) AS poles,
        SUM(CASE WHEN r.grid = 1 AND r.positionOrder = 1 THEN 1 ELSE 0 END) AS pole_to_wins,
        ROUND(100.0 * SUM(CASE WHEN r.grid = 1 AND r.positionOrder = 1 THEN 1 ELSE 0 END)
            / NULLIF(SUM(CASE WHEN r.grid = 1 THEN 1 ELSE 0 END), 0), 1) AS conversion_pct
    FROM results r
        JOIN drivers d ON r.driverId = d.driverId
    GROUP BY d.driverId
    HAVING poles >= 10
    ORDER BY conversion_pct DESC;
""")
df_wins

,driver,poles,pole_to_wins,conversion_pct
0,Max Verstappen,40,32,80.0
1,Alberto Ascari,14,9,64.3
2,Fernando Alonso,22,14,63.6
3,Michael Schumacher,68,40,58.8
4,Lewis Hamilton,104,61,58.7
5,Alain Prost,33,18,54.5
6,Sebastian Vettel,57,31,54.4
7,Nigel Mansell,32,17,53.1
8,James Hunt,14,7,50.0
9,Felipe Massa,16,8,50.0


In [ ]:
# 4. Constructors' championship dominance (win % per season)
df_wins = run_query("""
SELECT
    ra.year,
    c.name AS constructor,
    COUNT(*) AS races,
    SUM(CASE WHEN r.positionOrder = 1 THEN 1 ELSE 0 END) AS wins,
    ROUND(100.0 * SUM(CASE WHEN r.positionOrder = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS win_pct
FROM results r
JOIN races ra ON r.raceId = ra.raceId
JOIN constructors c ON r.constructorId = c.constructorId
WHERE r.grid > 0
GROUP BY ra.year, c.constructorId
HAVING wins > 0
ORDER BY ra.year DESC, wins DESC;
""")
df_wins

,year,constructor,races,wins,win_pct
0,2024,Red Bull,47,9,19.1
1,2024,McLaren,48,6,12.5
2,2024,Ferrari,47,5,10.6
3,2024,Mercedes,47,4,8.5
4,2023,Red Bull,42,21,50.0
...,...,...,...,...,...
272,1951,Alfa Romeo,30,5,16.7
273,1951,Ferrari,33,3,9.1
274,1951,Kurtis Kraft,17,1,5.9
275,1950,Alfa Romeo,22,6,27.3


In [22]:
# 5. DNF (Did Not Finish) rate by driver (modern era)
df_wins = run_query("""
    SELECT
        d.forename || ' ' || d.surname AS driver,
        COUNT(*) AS total_starts,
        SUM(CASE WHEN r.positionOrder > r.laps THEN 1 ELSE 0 END) AS dnfs,
        ROUND(100.0 * SUM(CASE WHEN r.positionOrder > r.laps THEN 1 ELSE 0 END) / COUNT(*), 1) AS dnf_pct
    FROM results r
        JOIN drivers d ON r.driverId = d.driverId
        JOIN races ra ON r.raceId = ra.raceId
    WHERE ra.year >= 2010
    GROUP BY d.driverId
    HAVING total_starts >= 30
    ORDER BY dnf_pct ASC;
""")
df_wins

,driver,total_starts,dnfs,dnf_pct
0,Lando Norris,128,4,3.1
1,Paul di Resta,59,2,3.4
2,Felipe Massa,155,6,3.9
3,Valtteri Bottas,247,11,4.5
4,Lewis Hamilton,304,14,4.6
5,Daniel Ricciardo,257,12,4.7
6,George Russell,128,6,4.7
7,Robert Kubica,42,2,4.8
8,Heikki Kovalainen,60,3,5.0
9,Felipe Nasr,40,2,5.0


2.3 Running SQL in Python

In [23]:
def run_query(sql, conn=conn):
    return pd.read_sql_query(sql, conn)

df_wins = run_query("""
    SELECT d.forename || ' ' || d.surname AS driver, COUNT(*) AS wins
    FROM results r JOIN drivers d ON r.driverId = d.driverId
    WHERE r.positionOrder = 1 GROUP BY d.driverId ORDER BY wins DESC LIMIT 15
""")
print(df_wins)

                driver  wins
0       Lewis Hamilton   105
1   Michael Schumacher    91
2       Max Verstappen    63
3     Sebastian Vettel    53
4          Alain Prost    51
5         Ayrton Senna    41
6      Fernando Alonso    32
7        Nigel Mansell    31
8       Jackie Stewart    27
9            Jim Clark    25
10          Niki Lauda    25
11         Juan Fangio    24
12       Nelson Piquet    23
13        Nico Rosberg    23
14          Damon Hill    22
